In [1]:
import pandas as pd
import numpy as np
import joblib

ROOT = r".."

FEATURES = [
    "followers",
    "following",
    "posts",
    "bio_length",
    "has_profile_picture",
    "is_private",
    "username_digit_ratio",
    "follower_following_ratio",
    "profile_completeness"
]

print("M18 - Risk Classification")
print("Loaded successfully!")

M18 - Risk Classification
Loaded successfully!


In [2]:
model = joblib.load(
    ROOT + r"\backend\trained_models\authentiscore_ensemble_v3.pkl"
)

df = pd.read_csv(
    ROOT + r"\data\processed\instagram_features.csv"
)

print("Model loaded!")
print("Dataset shape:", df.shape)

Model loaded!
Dataset shape: (696, 10)


In [3]:
def calculate_ml_risk(profile):

    X = pd.DataFrame(
        [[profile[feature] for feature in FEATURES]],
        columns=FEATURES
    )

    probability = model.predict_proba(X)[0][1]

    return round(float(probability * 100), 2)

In [4]:
def calculate_profile_rule_risk(profile):

    risk_score = 0
    signals = []

    # Low follower/following ratio
    if profile["following"] > 0:

        ratio = profile["followers"] / profile["following"]

        if ratio < 0.1:
            risk_score += 20

            signals.append({
                "rule": "low_follower_following_ratio",
                "weight": 20,
                "message": "Very low follower-to-following ratio"
            })

    # High username digit ratio
    if profile["username_digit_ratio"] > 0.5:

        risk_score += 15

        signals.append({
            "rule": "high_username_digit_ratio",
            "weight": 15,
            "message": "Username contains a high proportion of digits"
        })

    # Missing profile picture
    if profile["has_profile_picture"] == 0:

        risk_score += 20

        signals.append({
            "rule": "missing_profile_picture",
            "weight": 20,
            "message": "Profile picture is missing"
        })

    # Low profile completeness
    if profile["profile_completeness"] < 0.6:

        risk_score += 15

        signals.append({
            "rule": "low_profile_completeness",
            "weight": 15,
            "message": "Profile has low completeness"
        })

    # No posts
    if profile["posts"] == 0:

        risk_score += 10

        signals.append({
            "rule": "no_posts",
            "weight": 10,
            "message": "Profile has no posts"
        })

    # Empty bio
    if profile["bio_length"] == 0:

        risk_score += 10

        signals.append({
            "rule": "empty_bio",
            "weight": 10,
            "message": "Profile has no biography"
        })

    risk_score = min(risk_score, 100)

    return {
        "profile_rule_risk": risk_score,
        "signals": signals
    }

In [5]:
ML_WEIGHT = 0.60
NLP_WEIGHT = 0.25
RULE_WEIGHT = 0.15


def calculate_unified_risk(
    ml_risk,
    nlp_risk,
    rule_risk
):

    final_score = (
        ml_risk * ML_WEIGHT
        + nlp_risk * NLP_WEIGHT
        + rule_risk * RULE_WEIGHT
    )

    return round(
        min(max(final_score, 0), 100),
        2
    )

In [6]:
def classify_risk(score):

    if score < 30:
        return "Low Risk"

    elif score < 60:
        return "Medium Risk"

    elif score < 80:
        return "High Risk"

    else:
        return "Very High Risk"

In [7]:
def generate_risk_result(profile, nlp_risk=0):

    # ML component
    ml_risk = calculate_ml_risk(profile)

    # Rule component
    rule_result = calculate_profile_rule_risk(profile)

    rule_risk = rule_result["profile_rule_risk"]

    # Unified score
    final_score = calculate_unified_risk(
        ml_risk,
        nlp_risk,
        rule_risk
    )

    # Classification
    classification = classify_risk(final_score)

    result = {
        "ml_risk": ml_risk,
        "nlp_risk": nlp_risk,
        "rule_risk": rule_risk,
        "final_score": final_score,
        "classification": classification,
        "rule_signals": rule_result["signals"]
    }

    return result

In [8]:
profile = df.iloc[0]

result = generate_risk_result(
    profile,
    nlp_risk=0
)

print("=" * 60)
print("AUTHENTISCORE")
print("=" * 60)

for key, value in result.items():
    print(f"{key}: {value}")

AUTHENTISCORE
ml_risk: 3.67
nlp_risk: 0
rule_risk: 0
final_score: 2.2
classification: Low Risk
rule_signals: []


In [9]:
test_scores = [
    10,
    29.9,
    30,
    45,
    59.9,
    60,
    70,
    79.9,
    80,
    95
]

for score in test_scores:

    classification = classify_risk(score)

    print(
        f"Score: {score:>5} → {classification}"
    )

Score:    10 → Low Risk
Score:  29.9 → Low Risk
Score:    30 → Medium Risk
Score:    45 → Medium Risk
Score:  59.9 → Medium Risk
Score:    60 → High Risk
Score:    70 → High Risk
Score:  79.9 → High Risk
Score:    80 → Very High Risk
Score:    95 → Very High Risk
